# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jawad-ahmed-developer/flyRank_Internship_Tasks/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

For the Refresh / Content Opportunity Scoring lane, the raw warehouse table has a daily grain: **one row represents one content item for one client on one report date**.

For the ML feature table, these daily observations will later be aggregated to **one row per content item per client at a decision point**.

I will use **March 2026 as the development month**. The feature window will use information available before the future outcome window, so the model will not use future performance when constructing features.

The intended timeline is:

Previous observations → feature construction at decision time → future outcome.

In [37]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os
import getpass
import duckdb

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

if not HF_TOKEN:
    HF_TOKEN = getpass.getpass(
        "Paste your Hugging Face READ token (hf_...): "
    )

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "fact_daily":
        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
}

print("Connected to FlyRank warehouse.")
print("Development month: March 2026")

Paste your Hugging Face READ token (hf_...): ··········
Connected to FlyRank warehouse.
Development month: March 2026


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

For the first feature table, I will use five fields derived from information available at the decision point:

### Features

- **`imp_prev30`** — previous-30-day impressions; measures recent search visibility.
- **`clicks_prev30`** — previous-30-day clicks; measures recent organic traffic generated from search.
- **`avg_position_prev30`** — previous-30-day average search position; describes recent ranking performance.
- **`content_age_days`** — age of the content at the decision point; represents content maturity.
- **`days_since_last_update`** — days since the content was updated; represents freshness.

Each feature must be knowable before the future outcome occurs.

### Label / outcome

The eventual target will be a **future observed performance outcome**, such as whether the page experiences a meaningful decline in impressions during the future outcome window.

A decline signal calculated from the outcome window belongs to the label, not the feature set.

### Context

- `client_hash_id`
- `content_hash_id`
- `report_date`

These fields are needed to identify, group, join, and temporally align observations. They are not predictive features.

### Excluded

- **`trend_direction` and `trend_pct`** — excluded because they are derived trend signals and can contain information about the outcome being predicted.
- **Future-period impressions/clicks/position** — excluded because they are not available at decision time.
- **Client/content IDs as features** — excluded because pseudonymous identifiers do not represent meaningful page characteristics.
- **GA4 engagement fields** — excluded from the initial feature set because their availability is incomplete across the panel.
- **Fixed 90-day query-level signals** — excluded for this first contract because their time window can overlap the future outcome window and must first be temporally aligned.

In [38]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

FEATURES = [
    "imp_prev30",
    "clicks_prev30",
    "avg_position_prev30",
    "content_age_days",
    "days_since_last_update"
]

LABEL = [
    "future meaningful performance decline"
]

CONTEXT = [
    "client_hash_id",
    "content_hash_id",
    "report_date"
]

EXCLUDED = {
    "trend_direction": "derived trend/outcome information",
    "trend_pct": "derived trend/outcome information",
    "future performance": "not available at decision time",
    "client_hash_id": "identifier, not a predictive feature",
    "content_hash_id": "identifier, not a predictive feature",
    "GA4 engagement": "incomplete availability",
    "90-day query signals": "potential temporal overlap with outcome"
}

print("Features:", FEATURES)
print("Label:", LABEL)
print("Context:", CONTEXT)
print("Excluded:", EXCLUDED)

Features: ['imp_prev30', 'clicks_prev30', 'avg_position_prev30', 'content_age_days', 'days_since_last_update']
Label: ['future meaningful performance decline']
Context: ['client_hash_id', 'content_hash_id', 'report_date']
Excluded: {'trend_direction': 'derived trend/outcome information', 'trend_pct': 'derived trend/outcome information', 'future performance': 'not available at decision time', 'client_hash_id': 'identifier, not a predictive feature', 'content_hash_id': 'identifier, not a predictive feature', 'GA4 engagement': 'incomplete availability', '90-day query signals': 'potential temporal overlap with outcome'}


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

**Query #1:**   I will verify that the raw daily table has no duplicate combination of client, content item, and report date in the March 2026 development slice.

In [39]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

grain_check = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        COUNT(*) AS row_count
    FROM {TABLES["fact_daily"]}
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
    GROUP BY
        client_hash_id,
        content_hash_id,
        report_date
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print(grain_check)

print(
    "\nDuplicate grain combinations:",
    len(grain_check)
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Empty DataFrame
Columns: [client_hash_id, content_hash_id, report_date, row_count]
Index: []

Duplicate grain combinations: 0


**Query #2:**
I will verify the size and date coverage of the development slice directly from the warehouse rather than assuming that the documented dates are present in the selected partition.

In [40]:
march_check = con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        COUNT(DISTINCT client_hash_id) AS clients,
        COUNT(DISTINCT content_hash_id) AS content_items,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM {TABLES["fact_daily"]}
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
""").df()

print(march_check.to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 row_count  clients  content_items first_date  last_date
   9841378       55         331437 2026-03-01 2026-03-31


GA4 availability is represented explicitly by `ga4_data_available`. I will therefore use `IS TRUE` to count observations where GA4 data is confirmed to be available.

This check supports the decision to avoid relying on GA4 engagement features in the initial five-feature contract.

In [41]:
availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS ga4_available_rows
    FROM {TABLES["fact_daily"]}
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
      AND ga4_data_available IS TRUE
""").df()

print(availability_check.to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 ga4_available_rows
             413966


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This warehouse supports observed, directional, and decision-support analysis, but it has several limitations.

The panel is unbalanced, so clients have different amounts of historical data. Therefore, pages cannot automatically be assumed to have identical history.

GA4 data is not available for every observation, so a missing or unavailable GA4 value cannot be interpreted as zero engagement.

The query-level table uses a fixed 90-day window. Its fields therefore cannot automatically be treated as safe features for a future-outcome model because their observation window may overlap the outcome period.

The final month, June 2026, should be treated as a sealed outcome/test period rather than being used while developing the label or feature logic.

Finally, the data can show relationships between page characteristics and later performance, but it cannot establish that refreshing a page caused performance improvement or predict Google's ranking algorithm itself.

Therefore, the resulting system should be treated as decision support for prioritizing human SEO review, not as an automatic refresh decision or causal model.

In [42]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

limits_check = con.sql(f"""
    WITH client_history AS (
        SELECT
            client_hash_id,
            MIN(report_date) AS client_start,
            MAX(report_date) AS client_end
        FROM {TABLES["fact_daily"]}
        GROUP BY client_hash_id
    ),
    ga4 AS (
        SELECT
            COUNT(*) AS total_rows,
            SUM(
                CASE
                    WHEN ga4_data_available IS TRUE THEN 1
                    ELSE 0
                END
            ) AS ga4_available_rows
        FROM {TABLES["fact_daily"]}
        WHERE report_date >= DATE '2026-03-01'
          AND report_date < DATE '2026-04-01'
    ),
    warehouse_dates AS (
        SELECT
            MIN(report_date) AS warehouse_start,
            MAX(report_date) AS warehouse_end
        FROM {TABLES["fact_daily"]}
    )

    SELECT
        COUNT(*) AS clients,
        COUNT(DISTINCT client_start) AS distinct_client_start_dates,
        MIN(client_start) AS earliest_client_start,
        MAX(client_start) AS latest_client_start,
        ga4.total_rows,
        ga4.ga4_available_rows,
        warehouse_dates.warehouse_start,
        warehouse_dates.warehouse_end
    FROM client_history, ga4, warehouse_dates
    GROUP BY
        ga4.total_rows,
        ga4.ga4_available_rows,
        warehouse_dates.warehouse_start,
        warehouse_dates.warehouse_end
""").df()

print(limits_check.to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 clients  distinct_client_start_dates earliest_client_start latest_client_start  total_rows  ga4_available_rows warehouse_start warehouse_end
      70                           37            2025-01-27          2026-05-28     9841378            413966.0      2025-01-27    2026-06-30


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.